In [ ]:
# Notebook 4: PointNet++ APS to YCB-28 testing
# Set, paths, & imports

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm

PROJECT_DIR = Path("/content/drive/MyDrive/PointNet_APS_Project_V2")

SRC_DIR = PROJECT_DIR / "src"
MODELS_DIR = PROJECT_DIR / "models_final"
RESULTS_DIR = PROJECT_DIR / "results_final"
FIGURES_DIR = PROJECT_DIR / "figures_final"
METADATA_DIR = PROJECT_DIR / "metadata"

YCB_RESULTS_DIR = RESULTS_DIR / "ycb28_results"
YCB_CM_DIR = RESULTS_DIR / "ycb28_confusion_matrices"
YCB_FIG_DIR = FIGURES_DIR / "ycb28_figures"

for d in [YCB_RESULTS_DIR, YCB_CM_DIR, YCB_FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.append(str(SRC_DIR))

from pointnetpp_aps_utils import APSPointCloudDataset, PointNetPPClassifier, set_global_seed

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

CLASS_NAMES = ["box", "cylinder", "sphere"]

Mounted at /content/drive
Device: cuda


In [ ]:
# Load prepared YCB-28 point-cloud manifest

ycb_manifest_path = METADATA_DIR / "ycb28_pointcloud_manifest_bbox_norm.csv"
ycb_df = pd.read_csv(ycb_manifest_path)

print("YCB-28 manifest shape:", ycb_df.shape)
display(ycb_df.head())

print("\nClass counts:")
display(ycb_df["class_name"].value_counts())

print("\nObject counts:")
display(ycb_df.groupby(["class_name", "object_name"]).size().reset_index(name="count"))

print("\nExpected total:", 560)
print("Actual total:", len(ycb_df))

YCB-28 manifest shape: (560, 10)


,path,object_name,class_name,label,sample_id,variant,role,source_mesh,source_type,n_points
0,/content/drive/MyDrive/PointNet_APS_Project_V2...,003_cracker_box,box,0,0,ycb28,ycb28_test,/content/drive/MyDrive/PointNet_APS_Project_V2...,official_ycb_download,1000
1,/content/drive/MyDrive/PointNet_APS_Project_V2...,003_cracker_box,box,0,1,ycb28,ycb28_test,/content/drive/MyDrive/PointNet_APS_Project_V2...,official_ycb_download,1000
2,/content/drive/MyDrive/PointNet_APS_Project_V2...,003_cracker_box,box,0,2,ycb28,ycb28_test,/content/drive/MyDrive/PointNet_APS_Project_V2...,official_ycb_download,1000
3,/content/drive/MyDrive/PointNet_APS_Project_V2...,003_cracker_box,box,0,3,ycb28,ycb28_test,/content/drive/MyDrive/PointNet_APS_Project_V2...,official_ycb_download,1000
4,/content/drive/MyDrive/PointNet_APS_Project_V2...,003_cracker_box,box,0,4,ycb28,ycb28_test,/content/drive/MyDrive/PointNet_APS_Project_V2...,official_ycb_download,1000



Class counts:


,count
class_name,
sphere,220
box,180
cylinder,160



Object counts:


,class_name,object_name,count
0,box,003_cracker_box,20
1,box,004_sugar_box,20
2,box,008_pudding_box,20
3,box,009_gelatin_box,20
4,box,010_potted_meat_can,20
5,box,026_sponge,20
6,box,036_wood_block,20
7,box,061_foam_brick,20
8,box,077_rubiks_cube,20
9,cylinder,001_chips_can,20



Expected total: 560
Actual total: 560


In [ ]:
# Create YCB-28 test loader

N_POINTS = 1000
USE_NORMALS = True
BATCH_SIZE = 16

ycb_test_dataset = APSPointCloudDataset(
    ycb_df,
    n_points=N_POINTS,
    use_normals=USE_NORMALS
)

ycb_test_loader = DataLoader(
    ycb_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0
)

print("YCB test samples:", len(ycb_test_dataset))
print("Batches:", len(ycb_test_loader))

YCB test samples: 560
Batches: 35


In [ ]:
# Helper functions

import torch.nn.functional as F
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def load_pointnetpp_checkpoint(checkpoint_path, device):
    model = PointNetPPClassifier(num_classes=3, normal_channel=USE_NORMALS).to(device)

    checkpoint = torch.load(checkpoint_path, map_location=device)

    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            model.load_state_dict(checkpoint["model_state_dict"])
        elif "state_dict" in checkpoint:
            model.load_state_dict(checkpoint["state_dict"])
        else:
            model.load_state_dict(checkpoint)
    else:
        raise ValueError("Unsupported checkpoint format")

    model.eval()
    return model


def evaluate_model_on_ycb(model, loader, device):
    all_true = []
    all_pred = []
    all_conf = []

    model.eval()

    with torch.no_grad():
        for points, labels in loader:
            points = points.to(device)
            labels = labels.to(device)

            # Dataset usually gives [B, N, C], model expects [B, C, N]
            if points.shape[1] == N_POINTS:
                points = points.transpose(2, 1)

            logits = model(points)

            if isinstance(logits, tuple):
                logits = logits[0]

            probs = F.softmax(logits, dim=1)
            conf, preds = probs.max(dim=1)

            all_true.extend(labels.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_conf.extend(conf.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)

    return {
        "accuracy": acc,
        "true": np.array(all_true),
        "pred": np.array(all_pred),
        "confidence": np.array(all_conf)
    }

In [ ]:
# Check available trained PointNet++ checkpoints

TRAIN_CONDITIONS = ["clean_trained", "error_trained", "mixed_trained"]
RUN_IDS = list(range(1, 10))

checkpoint_records = []

for condition in TRAIN_CONDITIONS:
    for run_id in RUN_IDS:
        ckpt_path = MODELS_DIR / condition / f"{condition}_run_{run_id}_best.pth"
        checkpoint_records.append({
            "train_condition": condition,
            "run_id": run_id,
            "checkpoint_path": str(ckpt_path),
            "exists": ckpt_path.exists()
        })

checkpoint_df = pd.DataFrame(checkpoint_records)

display(checkpoint_df)

print("Checkpoint availability:")
display(checkpoint_df.groupby("train_condition")["exists"].sum())

print("\nMissing checkpoints:")
display(checkpoint_df[checkpoint_df["exists"] == False])

,train_condition,run_id,checkpoint_path,exists
0,clean_trained,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
1,clean_trained,2,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
2,clean_trained,3,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
3,clean_trained,4,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
4,clean_trained,5,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
5,clean_trained,6,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
6,clean_trained,7,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
7,clean_trained,8,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
8,clean_trained,9,/content/drive/MyDrive/PointNet_APS_Project_V2...,True
9,error_trained,1,/content/drive/MyDrive/PointNet_APS_Project_V2...,True


Checkpoint availability:


,exists
train_condition,
clean_trained,9
error_trained,9
mixed_trained,9



Missing checkpoints:


,train_condition,run_id,checkpoint_path,exists


In [ ]:
# Evaluate all trained PointNet++ models on YCB-28

ycb_summary_records = []
all_prediction_records = []

for _, row in checkpoint_df.iterrows():
    condition = row["train_condition"]
    run_id = int(row["run_id"])
    ckpt_path = Path(row["checkpoint_path"])

    if not ckpt_path.exists():
        print(f"Skipping missing checkpoint: {ckpt_path}")
        continue

    print(f"Evaluating {condition}, run {run_id}...")

    model = load_pointnetpp_checkpoint(ckpt_path, device)
    result = evaluate_model_on_ycb(model, ycb_test_loader, device)

    y_true = result["true"]
    y_pred = result["pred"]
    y_conf = result["confidence"]

    acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

    # Save confusion matrix
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = YCB_CM_DIR / f"{condition}_run_{run_id}_ycb28_confusion_matrix.csv"
    cm_df.to_csv(cm_path)

    # Save prediction records
    pred_df = ycb_df.copy().reset_index(drop=True)
    pred_df["true_label"] = y_true
    pred_df["pred_label"] = y_pred
    pred_df["true_class"] = [CLASS_NAMES[i] for i in y_true]
    pred_df["pred_class"] = [CLASS_NAMES[i] for i in y_pred]
    pred_df["confidence"] = y_conf
    pred_df["correct"] = pred_df["true_label"] == pred_df["pred_label"]
    pred_df["train_condition"] = condition
    pred_df["run_id"] = run_id

    pred_path = YCB_RESULTS_DIR / f"{condition}_run_{run_id}_ycb28_predictions.csv"
    pred_df.to_csv(pred_path, index=False)

    # Class-wise accuracy
    class_acc = {}
    for class_id, class_name in enumerate(CLASS_NAMES):
        mask = y_true == class_id
        class_acc[class_name] = accuracy_score(y_true[mask], y_pred[mask]) if mask.sum() > 0 else np.nan

    ycb_summary_records.append({
        "train_condition": condition,
        "run_id": run_id,
        "test_set": "ycb28",
        "test_accuracy": acc,
        "test_accuracy_percent": acc * 100,
        "correct_predictions": int((y_true == y_pred).sum()),
        "wrong_predictions": int((y_true != y_pred).sum()),
        "total_samples": len(y_true),
        "box_accuracy_percent": class_acc["box"] * 100,
        "cylinder_accuracy_percent": class_acc["cylinder"] * 100,
        "sphere_accuracy_percent": class_acc["sphere"] * 100,
        "confusion_matrix_path": str(cm_path),
        "prediction_path": str(pred_path)
    })

    all_prediction_records.append(pred_df)

ycb_summary_df = pd.DataFrame(ycb_summary_records)

summary_path = YCB_RESULTS_DIR / "pointnetpp_aps_to_ycb28_summary_runs_1_to_9.csv"
ycb_summary_df.to_csv(summary_path, index=False)

all_ycb_predictions_df = pd.concat(all_prediction_records, ignore_index=True)
all_predictions_path = YCB_RESULTS_DIR / "pointnetpp_aps_to_ycb28_all_predictions_runs_1_to_9.csv"
all_ycb_predictions_df.to_csv(all_predictions_path, index=False)

print("Saved YCB summary:")
print(summary_path)

print("\nSaved all predictions:")
print(all_predictions_path)

display(ycb_summary_df)

Evaluating clean_trained, run 1...
Evaluating clean_trained, run 2...
Evaluating clean_trained, run 3...
Evaluating clean_trained, run 4...
Evaluating clean_trained, run 5...
Evaluating clean_trained, run 6...
Evaluating clean_trained, run 7...
Evaluating clean_trained, run 8...
Evaluating clean_trained, run 9...
Evaluating error_trained, run 1...
Evaluating error_trained, run 2...
Evaluating error_trained, run 3...
Evaluating error_trained, run 4...
Evaluating error_trained, run 5...
Evaluating error_trained, run 6...
Evaluating error_trained, run 7...
Evaluating error_trained, run 8...
Evaluating error_trained, run 9...
Evaluating mixed_trained, run 1...
Evaluating mixed_trained, run 2...
Evaluating mixed_trained, run 3...
Evaluating mixed_trained, run 4...
Evaluating mixed_trained, run 5...
Evaluating mixed_trained, run 6...
Evaluating mixed_trained, run 7...
Evaluating mixed_trained, run 8...
Evaluating mixed_trained, run 9...
Saved YCB summary:
/content/drive/MyDrive/PointNet_APS_

,train_condition,run_id,test_set,test_accuracy,test_accuracy_percent,correct_predictions,wrong_predictions,total_samples,box_accuracy_percent,cylinder_accuracy_percent,sphere_accuracy_percent,confusion_matrix_path,prediction_path
0,clean_trained,1,ycb28,0.655357,65.535714,367,193,560,28.333333,86.875,80.454545,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
1,clean_trained,2,ycb28,0.628571,62.857143,352,208,560,20.000000,87.500,80.000000,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
2,clean_trained,3,ycb28,0.642857,64.285714,360,200,560,29.444444,66.875,90.909091,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
3,clean_trained,4,ycb28,0.653571,65.357143,366,194,560,95.000000,9.375,81.818182,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
4,clean_trained,5,ycb28,0.750000,75.000000,420,140,560,73.333333,75.000,76.363636,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
5,clean_trained,6,ycb28,0.641071,64.107143,359,201,560,48.888889,56.875,81.818182,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
6,clean_trained,7,ycb28,0.619643,61.964286,347,213,560,90.555556,21.250,68.181818,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
7,clean_trained,8,ycb28,0.644643,64.464286,361,199,560,40.000000,88.125,67.272727,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
8,clean_trained,9,ycb28,0.708929,70.892857,397,163,560,73.333333,56.250,79.545455,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...
9,error_trained,1,ycb28,0.714286,71.428571,400,160,560,79.444444,57.500,75.000000,/content/drive/MyDrive/PointNet_APS_Project_V2...,/content/drive/MyDrive/PointNet_APS_Project_V2...


In [ ]:
# Mean/std summary for YCB-28 results

ycb_mean_summary = (
    ycb_summary_df
    .groupby("train_condition")
    .agg(
        mean_accuracy_percent=("test_accuracy_percent", "mean"),
        std_accuracy_percent=("test_accuracy_percent", "std"),
        min_accuracy_percent=("test_accuracy_percent", "min"),
        max_accuracy_percent=("test_accuracy_percent", "max"),
        mean_box_accuracy_percent=("box_accuracy_percent", "mean"),
        mean_cylinder_accuracy_percent=("cylinder_accuracy_percent", "mean"),
        mean_sphere_accuracy_percent=("sphere_accuracy_percent", "mean")
    )
    .reset_index()
    .round(4)
)

display(ycb_mean_summary)

mean_summary_path = YCB_RESULTS_DIR / "pointnetpp_ycb28_mean_summary.csv"
ycb_mean_summary.to_csv(mean_summary_path, index=False)

print("Saved mean summary:")
print(mean_summary_path)

,train_condition,mean_accuracy_percent,std_accuracy_percent,min_accuracy_percent,max_accuracy_percent,mean_box_accuracy_percent,mean_cylinder_accuracy_percent,mean_sphere_accuracy_percent
0,clean_trained,66.0516,4.1921,61.9643,75.0000,55.4321,60.9028,78.4848
1,error_trained,67.3214,4.5763,58.3929,73.3929,42.9012,72.4306,83.5859
2,mixed_trained,67.1825,3.1451,63.3929,72.1429,45.8642,71.7361,81.3131


Saved mean summary:
/content/drive/MyDrive/PointNet_APS_Project_V2/results_final/ycb28_results/pointnetpp_ycb28_mean_summary.csv
